In [2]:
# ============================================================
# CELDA S1
# ============================================================
import pandas as pd

import os
REPO_URL = "https://github.com/Camilamop/RENABAP.git"

if os.path.exists('/content/RENABAP'):
    !cd /content/RENABAP && git fetch origin && git reset --hard origin/master
else:
    !git clone {REPO_URL} /content/RENABAP

!pip install geopandas --quiet

path_2023 = "/content/RENABAP/data/raw/RENABAP_2023.csv"
path_2022 = "/content/RENABAP/data/raw/RENABAP_2022.csv"
path_2018 = "/content/RENABAP/data/raw/RENABAP_2018.csv"

raw_2023 = pd.read_csv(path_2023, dtype=str, low_memory=False)
raw_2022 = pd.read_csv(path_2022, dtype=str, low_memory=False)
raw_2018 = pd.read_csv(path_2018, dtype=str, low_memory=False)

print("2023:", raw_2023.shape, "| 2022:", raw_2022.shape, "| 2018:", raw_2018.shape)

Cloning into '/content/RENABAP'...
remote: Enumerating objects: 165, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 165 (delta 60), reused 134 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (165/165), 8.49 MiB | 13.81 MiB/s, done.
Resolving deltas: 100% (60/60), done.
2023: (6467, 19) | 2022: (5687, 19) | 2018: (4416, 30)


In [3]:
# ============================================================
# CELDA S2 — Normalización de nombres de columnas de servicios

# ============================================================
MAPA_COLUMNAS = {
    2023: {
        "id_renabap": "id_renabap",
        "energia_electrica": "energia_electrica",
        "efluentes_cloacales": "efluentes_cloacales",
        "agua_corriente": "agua_corriente",
        "cocina": "cocina",
        "calefaccion": "calefaccion",
    },
    2022: {
        "id_renabap": "renabap_id",   # ojo: nombre distinto en 2022
        "energia_electrica": "energia_electrica",
        "efluentes_cloacales": "efluentes_cloacales",
        "agua_corriente": "agua_corriente",
        "cocina": "cocina",
        "calefaccion": "calefaccion",
    },
}

def normalizar_servicios(df, anio):
    m = MAPA_COLUMNAS[anio]
    out = df[[m["id_renabap"], m["energia_electrica"], m["efluentes_cloacales"],
              m["agua_corriente"], m["cocina"], m["calefaccion"]]].copy()
    out.columns = ["id_renabap", "energia_electrica", "efluentes_cloacales",
                   "agua_corriente", "cocina", "calefaccion"]
    out["anio_renabap"] = anio
    # "Sin Datos" es un no-dato disfrazado de string, no una categoría real
    out = out.replace("Sin Datos", pd.NA)
    return out

serv_2023 = normalizar_servicios(raw_2023, 2023)
serv_2022 = normalizar_servicios(raw_2022, 2022)
servicios_largo = pd.concat([serv_2022, serv_2023], ignore_index=True)

print(servicios_largo.groupby("anio_renabap").size())

anio_renabap
2022    5687
2023    6467
dtype: int64


In [4]:
# ============================================================
# CELDA S3 — Cantidad de datos y faltantes por servicio y año
# ============================================================
cols_servicio = ["energia_electrica", "efluentes_cloacales", "agua_corriente",
                  "cocina", "calefaccion"]

resumen_completitud = []
for anio, grupo in servicios_largo.groupby("anio_renabap"):
    n_total = len(grupo)
    for col in cols_servicio:
        n_validos = grupo[col].notna().sum()
        resumen_completitud.append({
            "anio": anio,
            "servicio": col,
            "n_total": n_total,
            "n_validos": n_validos,
            "n_faltantes": n_total - n_validos,
            "pct_faltante": round((n_total - n_validos) / n_total * 100, 2),
        })

df_completitud = pd.DataFrame(resumen_completitud)
df_completitud

,anio,servicio,n_total,n_validos,n_faltantes,pct_faltante
0,2022,energia_electrica,5687,5687,0,0.00
1,2022,efluentes_cloacales,5687,5687,0,0.00
2,2022,agua_corriente,5687,5687,0,0.00
3,2022,cocina,5687,5687,0,0.00
4,2022,calefaccion,5687,5041,646,11.36
5,2023,energia_electrica,6467,6467,0,0.00
6,2023,efluentes_cloacales,6467,6467,0,0.00
7,2023,agua_corriente,6467,6467,0,0.00
8,2023,cocina,6467,6467,0,0.00
9,2023,calefaccion,6467,5823,644,9.96


In [5]:
# ============================================================
# CELDA S4 — Valores únicos por servicio y año
# ============================================================
valores_unicos = []
for anio, grupo in servicios_largo.groupby("anio_renabap"):
    for col in cols_servicio:
        vc = grupo[col].value_counts(dropna=True)
        for valor, n in vc.items():
            valores_unicos.append({"anio": anio, "servicio": col, "valor": valor, "n": n})

df_valores_unicos = pd.DataFrame(valores_unicos).sort_values(["servicio", "anio", "n"], ascending=[True, True, False])
df_valores_unicos

,anio,servicio,valor,n
17,2022,agua_corriente,Conexión irregular a la red de agua,3223
18,2022,agua_corriente,Bomba de agua de pozo domiciliaria,930
19,2022,agua_corriente,Conexión formal a la red de agua con factura,526
20,2022,agua_corriente,Bomba de agua de pozo comunitaria,332
21,2022,agua_corriente,Camión cisterna,284
...,...,...,...,...
43,2023,energia_electrica,Conexión a la red con medidor compartido,85
44,2023,energia_electrica,Conexión regular a la red con medidor domicili...,53
45,2023,energia_electrica,Conexión regular a la red con medidor prepago,44
46,2023,energia_electrica,Generador eléctrico a combustión,7


In [6]:
# ============================================================
# CELDA S5 — Categorización tripartita: formal / informal / inexistente-precario
#
# CRITERIO (editable): me baso en las palabras que el propio RENABAP usa en
# sus etiquetas -> "formal"/"regular" = conexión legítima a la red (con o sin
# factura); "irregular"/"comunitario"/"compartido" = conectado a la red pero
# de manera no regularizada/no individualizada; todo lo que queda FUERA de
# la red (pozo, cisterna, garrafa, leña, generador, etc.) = inexistente/precario.
#
# ⚠️ DOS CASOS QUE TE DEJO MARCADOS PARA QUE DECIDAS (ver pregunta al final):
#   - Agua de pozo domiciliaria / cámara séptica: hoy están en "inexistente/
#     precario" porque están fuera de la red formal, pero en la literatura de
#     déficit habitacional (INDEC/NBI) a veces se consideran soluciones
#     "adecuadas" aunque no sean de red. Si querés tratarlas como una cuarta
#     categoría o subirlas a "informal", cambiá el diccionario abajo.
#   - "Energía eléctrica" para cocinar/calefaccionar: hoy en "inexistente/
#     precario" (fuera de la red de gas), aunque no es intrínsecamente
#     precaria si el suministro eléctrico es estable.
# ============================================================

CATEGORIZACION = {
    "energia_electrica": {
        "formal": [
            "Conexión formal a la red con medidor domiciliario con factura",
            "Conexión regular a la red con medidor domiciliario pero sin factura",
            "Conexión regular a la red con medidor domiciliario con consumo limitado",
            "Conexión regular a la red con medidor prepago",
        ],
        "informal": [
            "Conexión irregular a la red",
            "Conexión a la red con medidor comunitario",
            "Conexión a la red con medidor compartido",
        ],
        "inexistente_precario": [
            "No tiene conexión eléctrica",
            "Generador eléctrico a combustión",
            "Energía solar",
        ],
    },
    "agua_corriente": {
        "formal": [
            "Conexión formal a la red de agua con factura",
            "Conexión regular a la red de agua pero sin factura",
        ],
        "informal": [
            "Conexión irregular a la red de agua",
            "Canilla comunitaria dentro del barrio",
        ],
        "inexistente_precario": [
            "Bomba de agua de pozo domiciliaria",
            "Bomba de agua de pozo comunitaria",
            "Camión cisterna",
            "Acarreo de baldes/recipientes desde fuera del barrio",
            "Vertiente, arroyo, río o canal",
            "Cosecha/recolección de agua de lluvia",
        ],
    },
    "efluentes_cloacales": {
        "formal": [
            "Conexión formal a la red cloacal",
        ],
        "informal": [
            "Conexión irregular a la red cloacal",
            "Red cloacal conectada a la red pluvial",
        ],
        "inexistente_precario": [
            "Desagüe a cámara séptica y pozo ciego",
            "Desagüe sólo a pozo negro/ciego u hoyo",
            "Desagüe a intemperie o cuerpo de agua",
            "Baño seco",
            "Biodigestor para tratar efluentes cloacales",
        ],
    },
    "cocina": {
        "formal": [
            "Conexión formal a la red de gas con factura",
        ],
        "informal": [
            "Conexión irregular a la red de gas",
        ],
        "inexistente_precario": [
            "Gas en garrafa",
            "Leña o carbón",
            "Energía eléctrica",
        ],
    },
    "calefaccion": {
        "formal": [
            "Conexión formal a la red de gas con factura",
        ],
        "informal": [
            "Conexión irregular a la red de gas",
        ],
        "inexistente_precario": [
            "Gas en garrafa",
            "Leña o carbón",
            "Energía eléctrica",
            "Inexistente",
        ],
    },
}

def construir_lookup(categorizacion):
    lookup = {}
    for servicio, categorias in categorizacion.items():
        for cat, valores in categorias.items():
            for v in valores:
                lookup[(servicio, v)] = cat
    return lookup

LOOKUP = construir_lookup(CATEGORIZACION)

def categorizar(df, lookup, cols):
    out = df.copy()
    for col in cols:
        out[f"{col}_cat"] = out[col].apply(
            lambda v: lookup.get((col, v), pd.NA) if pd.notna(v) else pd.NA
        )
    return out

servicios_categorizado = categorizar(servicios_largo, LOOKUP, cols_servicio)

# chequeo de cobertura: valores que no matchean ningún diccionario
# (deberían ser 0 filas si el diccionario está completo)
for col in cols_servicio:
    no_mapeados = servicios_categorizado[
        servicios_categorizado[col].notna() & servicios_categorizado[f"{col}_cat"].isna()
    ][col].unique()
    if len(no_mapeados) > 0:
        print(f"⚠️ {col}: valores sin categorizar -> {no_mapeados}")

servicios_categorizado.head()

,id_renabap,energia_electrica,efluentes_cloacales,agua_corriente,cocina,calefaccion,anio_renabap,energia_electrica_cat,efluentes_cloacales_cat,agua_corriente_cat,cocina_cat,calefaccion_cat
0,1,Conexión regular a la red con medidor prepago,Desagüe a cámara séptica y pozo ciego,Bomba de agua de pozo domiciliaria,Gas en garrafa,<NA>,2022,formal,inexistente_precario,inexistente_precario,inexistente_precario,<NA>
1,2,Conexión irregular a la red,Desagüe a cámara séptica y pozo ciego,Conexión irregular a la red de agua,Gas en garrafa,Leña o carbón,2022,informal,inexistente_precario,informal,inexistente_precario,inexistente_precario
2,3,Conexión irregular a la red,Desagüe sólo a pozo negro/ciego u hoyo,Conexión formal a la red de agua con factura,Gas en garrafa,Leña o carbón,2022,informal,inexistente_precario,formal,inexistente_precario,inexistente_precario
3,4,Conexión irregular a la red,Desagüe sólo a pozo negro/ciego u hoyo,Conexión irregular a la red de agua,Gas en garrafa,Energía eléctrica,2022,informal,inexistente_precario,informal,inexistente_precario,inexistente_precario
4,5,Conexión irregular a la red,Desagüe a cámara séptica y pozo ciego,Bomba de agua de pozo domiciliaria,Gas en garrafa,<NA>,2022,informal,inexistente_precario,inexistente_precario,inexistente_precario,<NA>


In [7]:
# ============================================================
# CELDA S6 — 2018: diagnóstico de códigos, SIN categorizar todavía
# Los códigos numéricos no tienen diccionario disponible.
# Este bloque solo describe la situación, no inventa un mapeo.
# ============================================================
cols_2018 = ["Electricidad", "Disposición de excretas", "Acceso al agua",
             "Energía para cocinar", "Energía para calefaccionar"]

print("2018 — distribución de códigos (SIN etiqueta, pendiente de diccionario):\n")
for col in cols_2018:
    print(f"-- {col} --")
    print(raw_2018[col].value_counts(dropna=False).sort_index())
    print()

# Exploratorio (NO concluyente): si el mismo barrio aparece en 2018 y 2022,
# ¿el orden de los códigos numéricos de 2018 se correlaciona con alguna
# jerarquía formal→informal→precario visible en las categorías de texto
# de 2022? Sirve solo como pista para reconstruir el diccionario, no como
# fuente de verdad.
cruce = raw_2018[["id_renabap", "Electricidad"]].merge(
    serv_2022[["id_renabap", "energia_electrica"]],
    on="id_renabap", how="inner"
)
print("Barrios en común 2018-2022 para cruce exploratorio:", len(cruce))
print(pd.crosstab(cruce["Electricidad"], cruce["energia_electrica"]))

2018 — distribución de códigos (SIN etiqueta, pendiente de diccionario):

-- Electricidad --
Electricidad
1      22
2     235
3    1327
4    2744
5      32
6       1
7       1
8      52
9       2
Name: count, dtype: int64

-- Disposición de excretas --
Disposición de excretas
1      68
2      97
3     111
4     691
5    3355
6      79
7       5
8      10
Name: count, dtype: int64

-- Acceso al agua --
Acceso al agua
1      235
10     141
11      38
12      18
2      123
3     2619
4      692
5      243
6      104
7       30
8       56
9      117
Name: count, dtype: int64

-- Energía para cocinar --
Energía para cocinar
1      89
2       3
3      25
4    3950
5     335
6      14
Name: count, dtype: int64

-- Energía para calefaccionar --
Energía para calefaccionar
1      34
2       3
3    1491
4     340
5    1521
6     337
7     690
Name: count, dtype: int64

Barrios en común 2018-2022 para cruce exploratorio: 4279
energia_electrica  Conexión a la red con medidor compartido  \
Electrici

In [8]:
# ============================================================
# CELDA S7 (BONUS) — Índice de precariedad de servicios por barrio
# Cuenta cuántos de los 5 servicios están en "inexistente_precario"
# Útil para el análisis 1 (caracterización del universo) y para
# comparar barrios con y sin intervención (análisis 5)
# ============================================================
cat_cols = [f"{c}_cat" for c in cols_servicio]

servicios_categorizado["n_servicios_precarios"] = (
    servicios_categorizado[cat_cols] == "inexistente_precario"
).sum(axis=1)

servicios_categorizado["n_servicios_validos"] = servicios_categorizado[cat_cols].notna().sum(axis=1)

resumen_indice = servicios_categorizado.groupby("anio_renabap")["n_servicios_precarios"].describe()
resumen_indice

,count,mean,std,min,25%,50%,75%,max
anio_renabap,,,,,,,,
2022,5687.0,3.092316,0.687373,0.0,3.0,3.0,3.0,5.0
2023,6467.0,3.113654,0.679725,0.0,3.0,3.0,4.0,5.0


In [9]:
# ============================================================
# CELDA S8 (BONUS) — Comparación temporal 2022 → 2023 por barrio
# Directamente relevante para la hipótesis: ¿mejoró, empeoró o se
# mantuvo el acceso a cada servicio en los barrios con intervención?
# ============================================================
comparacion = servicios_categorizado[servicios_categorizado["anio_renabap"].isin([2022, 2023])]

pivot_list = []
for col in cols_servicio:
    piv = comparacion.pivot(index="id_renabap", columns="anio_renabap", values=f"{col}_cat")
    piv.columns = [f"{col}_2022", f"{col}_2023"]
    pivot_list.append(piv)

comparacion_wide = pd.concat(pivot_list, axis=1).reset_index()

for col in cols_servicio:
    c22, c23 = f"{col}_2022", f"{col}_2023"
    orden = {"inexistente_precario": 0, "informal": 1, "formal": 2}
    comparacion_wide[f"{col}_cambio"] = comparacion_wide.apply(
        lambda r: "mejoró" if pd.notna(r[c22]) and pd.notna(r[c23]) and orden.get(r[c23], -1) > orden.get(r[c22], -1)
        else ("empeoró" if pd.notna(r[c22]) and pd.notna(r[c23]) and orden.get(r[c23], -1) < orden.get(r[c22], -1)
        else ("igual" if pd.notna(r[c22]) and pd.notna(r[c23]) else pd.NA)),
        axis=1
    )

for col in cols_servicio:
    print(f"\n{col}:")
    print(comparacion_wide[f"{col}_cambio"].value_counts(dropna=False))


energia_electrica:
energia_electrica_cambio
igual    5654
<NA>      846
Name: count, dtype: int64

efluentes_cloacales:
efluentes_cloacales_cambio
igual    5654
<NA>      846
Name: count, dtype: int64

agua_corriente:
agua_corriente_cambio
igual    5654
<NA>      846
Name: count, dtype: int64

cocina:
cocina_cambio
igual     5653
<NA>       846
mejoró       1
Name: count, dtype: int64

calefaccion:
calefaccion_cambio
igual    5010
<NA>     1490
Name: count, dtype: int64
